In [46]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

# Preparation of Data Frames

In [47]:
# Reading the saved data pickle file
df  = pd.DataFrame()
# df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)
# df1 = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)


df=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl', compression=None)
df.index= pd.to_datetime(df.index)
df.Date= pd.to_datetime(df.Date)
df_vix=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix.csv')
df_vix = df_vix.fillna(0)
# df_vix["Date"] = df_vix.index
# df_vix.index= pd.to_datetime(df_vix.index)
df_vix.Date= pd.to_datetime(df_vix.Date)
# df_vix.index= pd.to_datetime(df_vix.index)


# del df['Date']
# del df_vix['Date']

df_vix_2010_2021=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix_2010_2021.csv')


In [48]:
df_vix

,Date,Open,High,Low,Close,Adj Close,Volume
0,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0
1,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0
2,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0
3,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0
4,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0
...,...,...,...,...,...,...,...
470,2021-08-06,17.459999,17.500000,16.139999,16.150000,16.150000,0
471,2021-08-10,16.820000,17.049999,16.340000,16.790001,16.790001,0
472,2021-08-11,16.809999,17.090000,16.070000,16.170000,16.170000,0
473,2021-08-12,16.330000,16.420000,15.490000,15.590000,15.590000,0


In [49]:
df = pd.merge(df_vix,df,how='outer', on='Date')
df=df.fillna(0)

# S&P 500

In [51]:
#if you get an error after executing the code, try adding below:
pd.core.common.is_list_like = pd.api.types.is_list_like

import pandas_datareader.data as web
import datetime

start = datetime.datetime(2019, 9, 25)
end = datetime.datetime(2021, 8, 13)
SP500  = pd.DataFrame()

SP500 = web.DataReader(['sp500'], 'fred', start, end)

In [52]:
SP500 = SP500.reset_index()

In [53]:
SP500

,DATE,sp500
0,2019-09-25,2984.87
1,2019-09-26,2977.62
2,2019-09-27,2961.79
3,2019-09-30,2976.74
4,2019-10-01,2940.25
...,...,...
488,2021-08-09,4432.35
489,2021-08-10,4436.75
490,2021-08-11,4447.70
491,2021-08-12,4460.83


In [54]:
df = df.join(SP500)

In [55]:
sp500_yesterday = df["sp500"].shift(periods=1)
df["sp500_ln_return"]=np.log(df["sp500"]/sp500_yesterday)

In [56]:
df.index = df.Date

In [57]:
df["vix"] = df["Adj Close"]
vix_yesterday = df["vix"].shift(periods=1)
df["vix_ln_return"]=np.log(df["vix"]/vix_yesterday)
df['sentiment'] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(10).mean()


In [58]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.index]
df['month'] = [d.strftime('%b') for d in df.index]
df['day'] = [d for d in df.index.day_name()]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.index]

In [59]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-08-13']

In [60]:
df

,Date,Open,High,Low,Close,Adj Close,Volume,compound,neg,neu,...,DATE,sp500,sp500_ln_return,vix,vix_ln_return,sentiment_ewm,year,month,day,month-year
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0.0,1.0,0.016,0.878,...,2019-09-25,2984.87,NaN,15.960000,NaN,86.991870,2019,Sep,Wednesday,Sep2019
2019-09-26,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0.0,1.0,0.023,0.855,...,2019-09-26,2977.62,-0.002432,16.070000,0.006869,85.439250,2019,Sep,Thursday,Sep2019
2019-09-27,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0.0,0.9997,0.035,0.86,...,2019-09-27,2961.79,-0.005331,17.219999,0.069117,81.623089,2019,Sep,Friday,Sep2019
2019-09-30,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0.0,1.0,0.026,0.842,...,2019-09-30,2976.74,0.005035,16.240000,-0.058594,82.174078,2019,Sep,Monday,Sep2019
2019-10-01,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0.0,0.9999,0.024,0.895,...,2019-10-01,2940.25,-0.012334,18.559999,0.133531,80.967511,2019,Oct,Tuesday,Oct2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-10,2021-08-10,16.820000,17.049999,16.340000,16.790001,16.790001,0.0,0.9999,0.025,0.837,...,2021-07-15,4360.03,-0.003268,16.790001,0.038863,83.901922,2021,Aug,Tuesday,Aug2021
2021-08-11,2021-08-11,16.809999,17.090000,16.070000,16.170000,16.170000,0.0,0.9999,0.025,0.85,...,2021-07-16,4327.16,-0.007568,16.170000,-0.037626,83.840063,2021,Aug,Wednesday,Aug2021
2021-08-12,2021-08-12,16.330000,16.420000,15.490000,15.590000,15.590000,0.0,0.9999,0.035,0.847,...,2021-07-19,4258.49,-0.015997,15.590000,-0.036528,83.229529,2021,Aug,Thursday,Aug2021


# VIX 2010 - 2021

In [61]:
# Import Data
df_vix_2010_2021=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/vix_2010_2021.csv')
df_vix_2010_2021['Date']= pd.to_datetime(df_vix_2010_2021['Date'])
df_vix_2010_2021.index= df_vix_2010_2021['Date']
print("df_vix_2010_2021['Date']="+str(type(df_vix_2010_2021['Date'])))
print("df_part3['Date']="+str(type(df_part3['Date'])))

df_vix_2010_2021["vix"] = df_vix_2010_2021["Adj Close"]#.apply(np.int)
df_vix_2010_2021["vix_ln_return"]=np.log(df_vix_2010_2021["vix"]/df_vix_2010_2021["vix"].shift(1))

import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()] 

df_vix_2010_2021['year'] = [d.year for d in df_vix_2010_2021.Date]
df_vix_2010_2021['month'] = [d.strftime('%b') for d in df_vix_2010_2021.Date]
df_vix_2010_2021['day'] = [d for d in df_vix_2010_2021.index.day_name()]
years = df_vix_2010_2021['year'].unique()
df_vix_2010_2021['month-year'] = [d.strftime('%b') + str(d.year) for d in df_vix_2010_2021.Date]

df_vix_2010_2021['Date']=<class 'pandas.core.series.Series'>
df_part3['Date']=<class 'pandas.core.series.Series'>


In [62]:
df_part3.fillna(0, inplace=True)
